In [25]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd

target_columns = ['1', '2', '3', '4', '5']

import json
import math
import re
import json
import numpy as np
from transformers import AutoTokenizer

pattern = re.compile(
    r"""['"]?
       overall\W+quality
       ['"]?
       \s*[:：]\s*
       ([1-5])
    """,
    re.IGNORECASE | re.VERBOSE
)

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-72B-Instruct",
    use_fast=True
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
def socreval_loader(X_path, y_path, dataset):
  raw_scores = []
  y_scores   = []
  logits_rows = []

  with open(X_path, 'r') as file:
      content_list = json.load(file)
  for i, item in enumerate(content_list):
    if item['judge'][-2].isdigit():
      raw_scores.append(int(item['judge'][-2]))
      target_logits = item['logprobs']['top_logprobs'][-3]
      logits_rows.append({str(k): max(v,math.log(1e-5)) for k, v in target_logits.items()})
    elif pattern.search(item['judge']): # score might exist in different places in qwen's response. Thus we design this to extract
      m = pattern.search(item['judge'])
      num_str = m.group(1)
      raw_scores.append(int(num_str))
      span = m.span(1)
      enc = tokenizer(item['judge'],
                return_offsets_mapping=True,
                return_tensors="pt")
      offsets = enc["offset_mapping"][0].tolist()
      token_idx = next(i for i,(s,e) in enumerate(offsets)
              if s <= span[0] < e)
      top_logprobs = item['logprobs']['top_logprobs']
      if token_idx < len(top_logprobs):
        target_logits = top_logprobs[token_idx]
        logits_rows.append({str(k): max(v,math.log(1e-5)) for k, v in target_logits.items()})
      else:
        logits_rows.append({})
        print(f"Warning: token_idx {token_idx} out of range")
    else:
      for line in item['judge'].splitlines():
            if "quality" in line.lower():
                print(f"--- item {i} ---", repr(line))
      print("Fails to find this score")
      raw_scores.append(np.nan)
      logits_rows.append({})

  raw_df = pd.DataFrame({'raw_score': raw_scores})
  raw_df.to_csv(f"SocREval_{dataset}_raw.csv", index=False)

  # print((pd.DataFrame(logits_rows)[[col for col in pd.DataFrame(logits_rows).columns if col not in ['1', '2', '3', '4', '5']]].applymap(lambda x: x > -11.512925)).sum().sum())

  X_df   = pd.DataFrame(logits_rows).reindex(columns=target_columns)
  X_df.fillna(math.log(1e-5), inplace=True)

  content_list = []
  with open(y_path, 'r') as file:
    for line in file:
        content_list.append(json.loads(line))
  y = pd.DataFrame(content_list)

  logits_df = pd.concat([X_df, y['human']], axis=1)
  logits_df.to_csv(f"SocREval_{dataset}_logits.csv", index=False)
  return logits_df




In [30]:
for dataset in ['cosmos', 'drop', 'esnli', 'gsm8k']:
  # json_path = f"/content/drive/MyDrive/Research/LLMCP/upload/judge/reasoning/local/dsr1/{dataset}_DeepSeek-R1-Distill-Qwen-32B.json"
  json_path = f"/content/drive/MyDrive/Research/LLMCP/upload/judge/reasoning/local/qwen/{dataset}_Qwen2.5-72B-Instruct.json"
  y_path = f"/content/drive/MyDrive/Research/LLMCP/upload/judge/reasoning/labels/ReasoningAnnotated_{dataset}_human.jsonl"
  logits_df = socreval_loader(json_path, y_path, dataset)

In [ ]:
with open(json_path, 'r') as file:
      content_list = json.load(file)

In [ ]:
def geval_loader(X_path, y_path, dataset):
  with open(json_path, 'r') as file:
      content_list = json.load(file)

  raw_scores = []
  y_scores   = []
  logits_rows = []

  for item in content_list:
    score = re.search(r'\d+', item['judge'])
    raw_scores.append(int(score.group()))

    target_logits = item['logprobs']['top_logprobs'][1]

    logits_rows.append({str(k): max(v,math.log(1e-5)) for k, v in target_logits.items()})

  raw_df = pd.DataFrame({'raw_score': raw_scores})
  raw_df.to_csv(f"GEval_{dataset}_raw.csv", index=False)

  X_df   = pd.DataFrame(logits_rows).reindex(columns=target_columns)

  content_list = []
  with open(y_path, 'r') as file:
    for line in file:
        content_list.append(json.loads(line))
  y = pd.DataFrame(content_list)

  logits_df = pd.concat([X_df, y['human']], axis=1)
  logits_df.to_csv(f"GEval_{dataset}_logits.csv", index=False)
  return logits_df

In [ ]:
for dataset in ['cosmos', 'drop', 'esnli', 'gsm8k']:
  json_path = f"/content/drive/MyDrive/Research/LLMCP/upload/judge/reasoning/local/qwen/GEval_{dataset}_Qwen2.5-72B-Instruct.json"
  y_path = f"/content/drive/MyDrive/Research/LLMCP/upload/judge/reasoning/labels/ReasoningAnnotated_{dataset}_human.jsonl"
  logits_df = geval_loader(json_path, y_path, dataset)